# Lab 05 · Ollama + Pinggy Server

Run in **Colab A**. Select a GPU runtime if available. Install and download before class.
This extends Lab 4 with two models: `embeddinggemma` finds evidence; `qwen3:4b` writes answers.
Keep Colab A running while the client uses it. Hardware availability and speed must be rehearsed; a single server is not assumed to serve the entire class concurrently.

### 1. Install Ollama

In [ ]:
# Install zstd
!apt-get update -qq
!apt-get install -y -qq zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Check installation
!ollama --version

### 2. Start Ollama locally

In [ ]:
import os, subprocess, time, requests

env = os.environ.copy()
env["OLLAMA_HOST"] = "127.0.0.1:11434"
log = open("/tmp/ollama.log", "a")
process = subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama did not start. Check /tmp/ollama.log")
print("Ollama is running on localhost:11434")

## 3. Download both models and check both APIs

In [ ]:
EMBED_MODEL = "embeddinggemma"
CHAT_MODEL = "qwen3:4b"
for model in [EMBED_MODEL, CHAT_MODEL]:
    subprocess.run(["ollama", "pull", model], check=True)

r = requests.post("http://127.0.0.1:11434/api/embed",
    json={"model": EMBED_MODEL, "input": ["Student services"]}, timeout=180)
r.raise_for_status()
print("Embedding dimension:", len(r.json()["embeddings"][0]))
r = requests.post("http://127.0.0.1:11434/api/chat", json={
    "model": CHAT_MODEL, "stream": False, "think": False,
    "messages": [{"role": "user", "content": "Reply with: Ready for Lab 5."}],
    "options": {"temperature": 0, "num_predict": 100}}, timeout=300)
r.raise_for_status()
print("Chat check:", r.json()["message"]["content"])
subprocess.run(["ollama", "ps"], check=True)

## 4. Open Pinggy (same workflow as Lab 4)
Run the terminal cell, then type:
```bash
ssh -p 443 -R0:localhost:11434 -t qr@free.pinggy.io "u:Host:localhost:11434"
```
Copy the HTTPS URL into the client notebook. Keep the terminal open.
The tunnel exposes the Ollama API: share only with the demo participants and close it after class.
If the URL changes, update the client and rerun its connection cell.

In [ ]:
%pip -q install colab-xterm
%load_ext colabxterm
%xterm

## After class
Press Ctrl+C in the tunnel terminal, then disconnect/delete the Colab runtime.
If generation is slow, run requests sequentially. If either API check fails, fix that check before opening the client.
Model/API reference: https://docs.ollama.com/api/chat and https://docs.ollama.com/api/embed.